In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')

In [3]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.5,0.5,0.5,0.00,0.0,None,0.3,6.80,2.0,1.0,0.00
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.0,0.5,0.5,2.81,0.0,None,0.3,16.86,1.0,1.0,2.75
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.0,0.5,0.5,1.00,0.0,None,0.3,8.30,1.0,1.0,0.00
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.0,0.5,0.5,0.00,0.0,None,0.3,9.30,2.0,1.0,0.00
4,2,2021-01-01 00:16:36,2021-01-01 00:16:40,N,2.0,265,265,3.0,0.00,-52.0,0.0,-0.5,0.00,0.0,None,-0.3,-52.80,3.0,1.0,0.00


In [4]:
import sklearn
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("first_experiment")

2026/10/28 17:58:10 INFO mlflow.tracking.fluent: Experiment with name 'first_experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/mlops-zoomcamp/mlruns/1', creation_time=1793210290293, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793210290293, lifecycle_stage='active', name='first_experiment', tags={}, trace_location=None, workspace='default'>

In [45]:
df = df[df.trip_type == 2]

In [46]:
df['duration'] = (
    df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
).dt.total_seconds() / 60

In [47]:
from sklearn.feature_extraction import DictVectorizer


In [48]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

train_dicts = df[categorical + numerical].to_dict(orient='records')
dv = DictVectorizer(sparse = False)
X_train = dv.fit_transform(train_dicts)

In [40]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error

target = 'duration'
y_train = df[target].values

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_train)
rmse = mean_squared_error(y_train, y_pred)

In [41]:
mean_squared_error(y_train, y_pred)

11478.089073147736

In [49]:
model = Lasso(0.001)
model.fit(X_train, y_train)
y_pred = model.predict(X_train)
mean_squared_error(y_train, y_pred)


11478.089073203995

In [50]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error

lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_train)

lasso = Lasso(alpha=0.001)
lasso.fit(X_train, y_train)
pred_lasso = lasso.predict(X_train)

mse_lr = mean_squared_error(y_train, pred_lr)
mse_lasso = mean_squared_error(y_train, pred_lasso)

print(f"LinearRegression: {mse_lr:.12f}")
print(f"Lasso:            {mse_lasso:.12f}")
print(f"Diferencia:       {mse_lasso - mse_lr:.12g}")

LinearRegression: 11478.089073147736
Lasso:            11478.089073203995
Diferencia:       5.62595232623e-08


In [51]:
import pickle

In [52]:
with open('models/lin_regresion.bin', 'wb') as f:
    pickle.dump(lr, f)